Data source: https://www.kaggle.com/competitions/iaa-ai-assisted-hackathon-2026-part-1

In [0]:
# Import / install libraries

%pip install xgboost openpyxl

#### Load data

In [0]:
from pyspark.sql import functions as F
import os

# FAST START: the contents TRAINING master was built and persisted in Motor_Bridget
# (workings/policies_contents_master.parquet, 2026-10-20) — read it directly instead of
# rebuilding. Raw datasets load lazily for the 2025 test master and submission steps.
workings_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings"
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"

df_policies_contents_master = spark.read.parquet(os.path.join(workings_dir, "policies_contents_master.parquet"))
print(f"Contents master: {df_policies_contents_master.count():,} rows, {len(df_policies_contents_master.columns)} columns (read from workings/)")

dataframes = {}
for file_name in sorted(f for f in os.listdir(data_dir) if f.endswith(".parquet")):
    name = file_name.replace(".parquet", "")
    dataframes[name] = spark.read.parquet(os.path.join(data_dir, file_name))
    globals()[f"df_{name}"] = dataframes[name]
print(f"Raw datasets loaded lazily: {', '.join(sorted(dataframes))}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# REBUILD THE CONTENTS MASTER WITHOUT THE RENEWED-ONLY FILTER (HUMAN DECISION
# 2026-10-21). The workings/policies_contents_master.parquet read in cell 5 keeps
# only renewal_decision == 'renewed' (~83% of rows), while the 2025 test files contain
# every policy regardless of renewal outcome — a train/test population mismatch
# flagged after the Kaggle v008-vs-v009 A/B (the renewed-only buildings line scored
# below the constant baseline). This cell rebuilds the contents master from the raw
# data with the SAME joins as Motor_Bridget cell 10 but WITHOUT the filter, and
# rebinds df_policies_contents_master. The workings/ parquet is intentionally NOT
# overwritten — the EDA notebooks still read the renewed-only original.

CLAIM_FIRST_COLS = ["claim_id", "loss_date", "report_date", "status", "repairer_id", "third_party_id"]

wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

pol = dataframes["policies_contents"]        # NO renewal_decision filter — all policies

cl = (dataframes["claims"]
      .filter(F.col("line") == "contents")
      .withColumn("_claim_year", F.year("loss_date"))
      .withColumn("_rn", F.row_number().over(
          Window.partitionBy("policy_id", "_claim_year").orderBy("claim_id"))))

cl_agg = (cl.groupBy("policy_id", "_claim_year")
          .agg(
              F.sum("paid_to_date").alias("paid_to_date"),
              F.sum("incurred_to_date").alias("incurred_to_date"),
              F.sum("recoveries").alias("recoveries"),
              F.count(F.lit(1)).alias("n_claims"),
              F.collect_set("peril").alias("_perils"),
              *[F.max(F.when(F.col("_rn") == 1, F.col(c))).alias(c) for c in CLAIM_FIRST_COLS])
          .withColumn("peril",
                      F.when(F.size(F.col("_perils")) > 1, F.lit("multiple"))
                       .otherwise(F.element_at(F.col("_perils"), 1)))
          .drop("_perils")
          .withColumnRenamed("policy_id", "_cl_policy_id"))

df_policies_contents_master = (
    pol
    .join(wx,
          (F.col("address_id") == F.col("_wx_address_id")) &
          (F.year(F.col("period_start")) == F.col("_wx_year")),
          "left")
    .drop("_wx_address_id", "_wx_year")
    .join(dataframes["customers"], "customer_id", "left")
    .join(dataframes["addresses"], "address_id", "left")
    .join(cl_agg,
          (F.col("policy_id") == F.col("_cl_policy_id")) &
          (F.year(F.col("period_start")) == F.col("_claim_year")),
          "left")
    .drop("_cl_policy_id", "_claim_year")
    .withColumn("claim_indicator", F.when(F.col("n_claims") >= 1, 1).otherwise(0))
    .withColumn("incurred_net_of_recoveries", F.col("incurred_to_date") - F.col("recoveries"))
    .withColumn("ultimate_claims", F.col("incurred_to_date") - F.col("recoveries"))
    .withColumn("_days_total", F.datediff("period_end", "period_start") + 1)
    .withColumn("_year_end", F.make_date(F.year("period_start"), F.lit(12), F.lit(31)))
    .withColumn("_days_earned", F.least(F.col("_days_total"),
                                        F.datediff(F.col("_year_end"), "period_start") + 1))
    .withColumn("_year_days", F.datediff("_year_end",
                                         F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
    .withColumn("earned_premium",
                F.round(F.col("written_premium") * F.col("_days_earned") / F.col("_year_days"), 2))
    .drop("_days_total", "_year_end", "_days_earned", "_year_days")
)

# Sanity check: row count must equal ALL contents policy rows (no join fan-out)
n_pol_all = dataframes["policies_contents"].count()
s = df_policies_contents_master.agg(
    F.count(F.lit(1)).alias("rows"),
    F.sum(F.when(F.col("n_claims").isNotNull(), 1).otherwise(0)).alias("with_claims"),
    F.round(F.sum("incurred_to_date") - F.sum("recoveries"), 0).alias("chk_net"),
    F.round(F.sum("ultimate_claims"), 0).alias("ult_total"),
).first()
print(f"UNFILTERED contents master: {s['rows']:,} rows (all policies {n_pol_all:,}, match: {s['rows'] == n_pol_all}) | "
      f"claims matched: {s['with_claims']:,} ({100.0 * s['with_claims'] / s['rows']:.1f}%) | "
      f"ultimate_claims total: {s['ult_total']:,.0f} (consistent: {s['ult_total'] == s['chk_net']})")

#### Contents feature selection

In [0]:
# Feature selection for the contents Tweedie XGBoost — HUMAN DECISION (2026-10-20):
# expert-judgement list provided by Bridget (1 = use, 0 = exclude; see chat).
# Target = ultimate_claims (nulls -> 0 for claim-free policy-years).
# 'Year' is derived from period_start. All claims-derived columns are excluded as
# features (leakage); IDs are kept only for GroupKFold grouping and prediction tracking.
# NOTE: the seven weather features were left BLANK in the provided list — they are
# INCLUDED here (consistent with the motor model and directly relevant to property
# risk); trivial to drop if that was not the intent.

CONTENTS_FEATURES = [
    "Year",                                      # derived from period_start
    # exposure (HUMAN DECISION: written_premium and discount_applied excluded)
    "exposure",
    # policy & cover
    "policyholder_age", "sum_insured", "tenure", "security_level", "high_value_items",
    "accidental_damage_cover", "works_from_home", "lodgers", "bicycle_cover",
    "voluntary_excess", "compulsory_excess",
    # policy admin
    "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
    "days_quote_to_inception",
    # customer
    "occupation", "marital_status", "household_size", "credit_band", "prior_claims_5y",
    "licence_points", "years_at_address",
    # address & property
    "branch_id", "marketing_segment", "region", "urban_density", "flood_zone",
    "distance_to_river_km", "elevation_m", "soil_type", "year_built", "property_type",
    "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",
    "energy_rating", "solar_panels",
    # weather (blank in the provided list — included, see note above)
    "hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index",
    "wildfire_index", "rainfall_mm", "mean_temperature_c",
]

TARGET = "ultimate_claims"
GROUP_COL = "customer_id"              # GroupKFold grouping: all years of one customer stay together
TRACK_COLS = ["policy_id", "row_id"]   # for attaching predictions; never model inputs
EVAL_COL = "claim_indicator"           # diagnostic only (decile claim rates), not a feature

# Every listed column must exist in the master (Year is derived here, not in the master)
_missing = [c for c in CONTENTS_FEATURES + [TARGET, GROUP_COL, EVAL_COL, *TRACK_COLS]
            if c != "Year" and c not in df_policies_contents_master.columns]
assert not _missing, f"Listed columns missing from the master: {_missing}"

contents_model = (df_policies_contents_master
               .withColumn("Year", F.year("period_start"))
               # EXPOSURE FIX (2026-10-21, user instruction): the raw exposure column mixes
               # scales (EDA: 2,961 of 599,110 rows > 1, up to 365.0 — day counts stored as
               # year fractions, same issue as motor and buildings). Recompute from the
               # period dates with the fixed 365 divisor the user specified for motor:
               # (period_end - period_start + 1) / 365.
               .withColumn("exposure", (F.datediff("period_end", "period_start") + F.lit(1)) / F.lit(365))
               .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0)))   # no claims -> 0
               .select(*TRACK_COLS, GROUP_COL, *CONTENTS_FEATURES, TARGET, EVAL_COL))

# MODE IMPUTATION (2026-10-21, user instruction): 11 of the 49 features carry ~2.5%
# nulls (flood_zone 2.9%; security_level, high_value_items, years_at_address,
# marital_status, energy_rating, occupation, distance_to_river_km, soil_type,
# roof_type, household_size). Impute each with the TRAINING mode; the modes are learned
# here and reused unchanged on the 2025 test master (cell 13) so train/test stay
# aligned. Numeric features are also imputed with their mode per instruction —
# median is the more usual choice for continuous features, flagged for review.
IMPUTE_COLS = ["flood_zone", "security_level", "high_value_items", "years_at_address",
               "marital_status", "energy_rating", "occupation", "distance_to_river_km",
               "soil_type", "roof_type", "household_size"]
MODES = {}
for c in IMPUTE_COLS:
    # Exclude nulls when taking the mode — for near-continuous features (e.g.
    # distance_to_river_km) every distinct value can be unique, making the NULL group
    # the single largest "value"; the mode must be a real observed value.
    MODES[c] = (contents_model.where(F.col(c).isNotNull())
                .groupBy(c).count()
                .orderBy(F.desc("count"), F.asc(c)).first()[c])
    assert MODES[c] is not None, f"mode of {c} came back None"
contents_model = contents_model.fillna(MODES)
print("Mode imputation (training modes, reused on the 2025 test master in cell 13):")
for c, m in MODES.items():
    print(f"  {c:<22} -> {m}")

# One pass over the assembled dataset: size, target behaviour, feature missingness
s = contents_model.agg(
    F.count(F.lit(1)).alias("_rows"),
    F.sum(F.when(F.col(TARGET) > 0, 1).otherwise(0)).alias("_claim_rows"),
    F.round(F.sum(TARGET), 2).alias("_target_sum"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in CONTENTS_FEATURES],
).first()

n_rows = s["_rows"]
print(f"Contents modeling dataset: {n_rows:,} rows | {len(CONTENTS_FEATURES)} features | target = {TARGET} (nulls->0)")
print(f"Rows with target > 0: {s['_claim_rows']:,} ({100.0*s['_claim_rows']/n_rows:.1f}%) | total target sum: {s['_target_sum']:,.0f}")
print(f"Zero-target rows: {n_rows - s['_claim_rows']:,} ({100.0*(n_rows - s['_claim_rows'])/n_rows:.1f}%)")

feat_nulls = {c: s[c] for c in CONTENTS_FEATURES if s[c] and s[c] > 0}
print("\nFeatures with missing values (left as-is — XGBoost handles them natively):")
for c, k in sorted(feat_nulls.items(), key=lambda x: -x[1]):
    print(f"  {c:<22} {100.0*k/n_rows:6.2f}%")

from collections import Counter
_dtypes = {c: t for c, t in contents_model.dtypes if c in CONTENTS_FEATURES}
print(f"\nFeature dtypes: {dict(Counter(_dtypes.values()))}")

display(contents_model.limit(10))

#### Tweedie XGBoost modeling (contents)

4-fold GroupKFold (grouped by customer_id) Tweedie XGBoost on the contents master, mirroring the motor pipeline. The Tweedie variance power is fixed at 1.5 to match the hackathon scoring, and models are evaluated with mean Tweedie deviance at p=1.5 — the same loss form the competition uses.

In [0]:
import pandas as pd

# Convert the contents modeling dataset to pandas for XGBoost training:
# strings -> category (used natively via enable_categorical=True), booleans -> int.
# The same conversions, with category vocabularies learned from TRAINING only at
# scoring time, must be reused when scoring the test file so features stay aligned.

pdf = contents_model.toPandas()

CAT_FEATURES = [c for c in CONTENTS_FEATURES if str(pdf[c].dtype) == "object"]
BOOL_FEATURES = [c for c in CONTENTS_FEATURES if pdf[c].dtype == bool]
for c in CAT_FEATURES:
    pdf[c] = pdf[c].astype("category")
for c in BOOL_FEATURES:
    pdf[c] = pdf[c].astype("int8")

X = pdf[CONTENTS_FEATURES]
y = pdf[TARGET].to_numpy(dtype=float)
w = pdf["exposure"].to_numpy(dtype=float)   # exposure weight: Tweedie pure premium model
groups = pdf[GROUP_COL]

print(f"Training matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Categorical ({len(CAT_FEATURES)}): {', '.join(CAT_FEATURES)}")
print(f"Boolean -> int ({len(BOOL_FEATURES)}): {', '.join(BOOL_FEATURES)}")
print(f"Exposure weight: min {w.min():.4f} | max {w.max():.2f} | zero: {(w == 0).sum():,} | >1: {(w > 1).sum():,} (known data issue, EDA later)")
display(pdf.head(5))

In [0]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold

# GroupKFold (4 folds): all policy-years of the same customer stay in one fold,
# so no customer is ever in both train and validation (avoids leakage across years).
# The hackathon scores with Tweedie power 1.5, so tweedie_variance_power is FIXED
# at 1.5 and each fold is evaluated with mean Tweedie deviance at p=1.5 — the same
# loss form the competition uses (XGBoost's built-in metric is a log-likelihood,
# not deviance).

POWER = 1.5          # Tweedie variance power — matches hackathon scoring (HUMAN NOTED)
N_FOLDS = 4

XGB_PARAMS = dict(
    objective="reg:tweedie",
    tweedie_variance_power=POWER,
    tree_method="hist",
    enable_categorical=True,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

def tweedie_deviance(y_true, y_pred, p=POWER):
    """Mean Tweedie unit deviance (1<p<2) — the loss form the hackathon scores on (p=1.5)."""
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0.0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    unit = 2.0 * (
        np.power(y_true, 2.0 - p) / ((1.0 - p) * (2.0 - p))
        - y_true * np.power(y_pred, 1.0 - p) / (1.0 - p)
        + np.power(y_pred, 2.0 - p) / (2.0 - p)
    )
    return float(np.mean(unit))

gkf = GroupKFold(n_splits=N_FOLDS)
oof = pd.Series(np.nan, index=pdf.index, name="pred")
fold_rows = []
gain_total = {}   # feature -> summed gain across the 4 fold models

for k, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X.iloc[tr], y[tr], sample_weight=w[tr])
    pred = model.predict(X.iloc[va])
    oof.iloc[va] = pred

    score = model.get_booster().get_score(importance_type="gain")
    for f, g in score.items():
        gain_total[f] = gain_total.get(f, 0.0) + g

    dev = tweedie_deviance(y[va], pred)
    fold_rows.append({"fold": k, "train_rows": len(tr), "val_rows": len(va),
                      "tweedie_deviance_p1.5": dev,
                      "mean_pred": float(pred.mean()), "mean_actual": float(y[va].mean())})
    print(f"Fold {k}: {len(tr):,} train | {len(va):,} val | deviance(p=1.5) {dev:.4f} | "
          f"mean pred {pred.mean():.3f} vs actual {y[va].mean():.3f}")

fold_summary = pd.DataFrame(fold_rows)
print(f"\nMean Tweedie deviance (p=1.5) across folds: {fold_summary['tweedie_deviance_p1.5'].mean():.4f}")
display(fold_summary)

In [0]:
import os

# Out-of-fold (OOF) evaluation: every row is scored by a fold model that never saw it.
# Decile check: rows sorted into deciles by predicted pure premium — actual claims and
# claim rate should rise across deciles. Feature importance (gain) feeds the
# EDA/refinement loop. All results exported to Excel for review.

excel_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files"

pdf["oof_pred"] = oof
pdf["pred_decile"] = pd.qcut(pdf["oof_pred"], 10, labels=False, duplicates="drop") + 1

decile_tbl = (pdf.groupby("pred_decile")
              .agg(n=("oof_pred", "size"),
                   mean_pred=("oof_pred", "mean"),
                   mean_actual=(TARGET, "mean"),
                   claim_rate=(EVAL_COL, "mean"))
              .reset_index())

mono = (decile_tbl["mean_actual"].diff().dropna() >= 0).mean()
lift = decile_tbl["mean_actual"].iloc[-1] / pdf[TARGET].mean()
print(f"Decile monotonicity (share of steps non-decreasing in actual claims): {mono:.0%}")
print(f"Top-decile lift (mean actual claims, top decile vs overall): {lift:.2f}x")
display(decile_tbl)

imp = (pd.DataFrame({"feature": list(gain_total.keys()), "total_gain": list(gain_total.values())})
       .sort_values("total_gain", ascending=False).reset_index(drop=True))
imp["gain_share"] = imp["total_gain"] / imp["total_gain"].sum()
print("Top 15 features by gain:")
display(imp.head(15))

out_path = os.path.join(excel_dir, "contents_tweedie_xgb_results.xlsx")
with pd.ExcelWriter(out_path) as xw:
    decile_tbl.to_excel(xw, sheet_name="decile_lift", index=False)
    imp.to_excel(xw, sheet_name="feature_importance", index=False)
    fold_summary.to_excel(xw, sheet_name="cv_folds", index=False)
print(f"Results exported to: {out_path}")

#### Test-file scoring & submission (contents)

Refit the contents Tweedie XGBoost on 100% of the training rows (the CV run was evaluation only), build the 2025 test contents master with the same joins as training, predict ultimate_claims, and fill the C-prefix rows of the sample_submission skeleton. Motor/buildings rows (and any zero predictions) keep the dummy value 10 — the scorer does not accept zeros.

In [0]:
# Build the contents TEST master for the 2025 policies, mirroring the training master
# joins: weather (address_id + policy year), customers (customer_id), addresses
# (address_id). The test file has no claims data and no renewal_decision, so there is
# no claims join and no renewal filter — every test row is scored. Since the
# 2026-10-21 rebuild, TRAINING also uses all policies (unfiltered-master rebuild cell
# above), so train and test populations are now aligned.

wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

test_contents_master = (
    dataframes["test_contents"]
    .join(wx,
          (F.col("address_id") == F.col("_wx_address_id")) &
          (F.year(F.col("period_start")) == F.col("_wx_year")),
          "left")
    .drop("_wx_address_id", "_wx_year")
    .join(dataframes["customers"], "customer_id", "left")
    .join(dataframes["addresses"], "address_id", "left")
)

test_contents_model = (test_contents_master
    .withColumn("Year", F.year("period_start"))
    # Same exposure fix as training (cell 7): recompute from the period dates with
    # the fixed 365 divisor, not the raw column (2025 is not a leap year)
    .withColumn("exposure", (F.datediff("period_end", "period_start") + F.lit(1)) / F.lit(365))
    .select("row_id", "policy_id", "customer_id", *CONTENTS_FEATURES))

# Every training feature must be present in the test master
_t_missing = [c for c in CONTENTS_FEATURES if c != "Year" and c not in test_contents_model.columns]
assert not _t_missing, f"Features missing from the test master: {_t_missing}"

test_pdf = test_contents_model.toPandas()
print(f"Test contents master: {len(test_pdf):,} rows | {len(CONTENTS_FEATURES)} features (same as training)")
print(f"Customer-feature nulls (join gap): {test_pdf['occupation'].isna().mean():.1%}")

# Mode imputation on the test master using the TRAINING modes from cell 7 — the same
# treatment as training. Test join gaps also get the training mode, keeping the test
# feature distributions aligned with training.
for c, m in MODES.items():
    test_pdf[c] = test_pdf[c].fillna(m)
print(f"Mode imputation applied with {len(MODES)} training modes | "
      f"remaining nulls in imputed columns: {int(test_pdf[IMPUTE_COLS].isna().sum().sum())}")

# Align categoricals with the TRAINING vocabularies; unseen test levels -> NaN.
# Booleans -> int, exactly as in the training prep.
X_test = test_pdf[CONTENTS_FEATURES].copy()
_new_levels = {}
for c in CAT_FEATURES:
    cats = pdf[c].cat.categories
    _new_levels[c] = int((~test_pdf[c].dropna().isin(cats)).sum())
    X_test[c] = pd.Categorical(test_pdf[c], categories=cats)
for c in BOOL_FEATURES:
    X_test[c] = X_test[c].astype("int8")

_new = {c: k for c, k in _new_levels.items() if k > 0}
print(f"Unseen categorical levels (mapped to NaN): {_new if _new else 'none'}")
display(X_test.head(5))

In [0]:
import os
import numpy as np

# FINAL model: refit the CV configuration on 100% of the training rows — the CV run
# was evaluation only; the final model uses all training data before scoring the 2025
# contents test rows (HUMAN INSTRUCTION 2026-10-20: predict test_contents and add the
# results to the sample_submission skeleton).
final_model = xgb.XGBRegressor(**XGB_PARAMS)
final_model.fit(X, y, sample_weight=w)
test_pred = final_model.predict(X_test)

# Submission on the sample_submission skeleton (row_id + prediction). Contents rows
# (row_id prefix C) get the model prediction; motor/buildings rows have no model in
# this file and the scorer rejects zeros, so they — and any rounded-zero contents
# prediction — keep the dummy value 0.01 (HUMAN DECISION 2026-10-21: the previous
# dummy of 10 was an arbitrary positive level; 0.01 keeps every prediction strictly
# positive with minimal distortion).
DUMMY = 0.01
assert test_pdf["row_id"].is_unique, "test_contents row_id is not unique"

sub = dataframes["sample_submission"].select("row_id").toPandas()
contents_map = pd.DataFrame({"row_id": test_pdf["row_id"], "prediction": np.round(test_pred, 2)})
sub = sub.merge(contents_map, on="row_id", how="left")
sub["prediction"] = sub["prediction"].fillna(DUMMY)
sub.loc[sub["prediction"] == 0, "prediction"] = DUMMY

# Sanity checks: full skeleton, no nulls or zeros, every C row scored by the model
assert len(sub) == 626_639, f"unexpected row count: {len(sub):,}"
assert sub["prediction"].isna().sum() == 0 and (sub["prediction"] <= 0).sum() == 0

pref = sub["row_id"].str[0]
c_mask = pref == "C"
assert c_mask.sum() == len(test_pdf), "C-row count does not match test_contents rows"
print(f"Submission rows: {len(sub):,} | by line: {pref.value_counts().to_dict()}")
print(f"Contents rows scored by the model: {c_mask.sum():,} | "
      f"mean pred {sub.loc[c_mask, 'prediction'].mean():.2f} (training mean claims/row 77.8) | "
      f"min {sub.loc[c_mask, 'prediction'].min():.2f} | max {sub.loc[c_mask, 'prediction'].max():.2f}")
print(f"Dummy rows (M/B lines, no model in this file): {(~c_mask).sum():,} | "
      f"contents zero-preds replaced by dummy: {int(sub.loc[c_mask, 'prediction'].eq(DUMMY).sum())}")

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"
os.makedirs(sub_dir, exist_ok=True)
sub_path = os.path.join(sub_dir, "contents_tweedie_xgb_submission_v003.csv")  # v003 = all policies (no renewed-only filter) + DUMMY 0.01 (v001/v002 kept on disk)
sub.to_csv(sub_path, index=False)
print(f"\nSubmission saved to: {sub_path}")
display(sub.head(10))

#### Combined submission v8 (motor + buildings + contents)

Assemble the full submission with all three lines model-scored by Bridget's own models: M rows from the motor Tweedie XGBoost v003 file (all policies, written_premium dropped, DUMMY 0.01), C rows from the contents model v003 file built above (all policies, DUMMY 0.01), B rows from the buildings model v004 file (all policies, PRED_FLOOR removed, DUMMY 0.01). v008/v009 are left untouched as the previous submissions; this writes a new v010 file. Each line's rows are taken directly from its own submission file before saving.

In [0]:
import os
import pandas as pd

# Combined v010 (HUMAN DECISIONS 2026-10-21): M <- motor model v003 (all policies,
# written_premium dropped, DUMMY 0.01), C <- contents model v003 (all policies,
# DUMMY 0.01 — this notebook), B <- buildings model v004 (all policies, PRED_FLOOR
# removed, DUMMY 0.01). Sources are read from disk so the assembly is deterministic
# and reproducible after any restart. v008/v009 are kept untouched as the previous
# submissions.
sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

motor_v3 = pd.read_csv(os.path.join(sub_dir, "motor_tweedie_xgb_submission_v003.csv"))        # M model rows
contents_v3 = pd.read_csv(os.path.join(sub_dir, "contents_tweedie_xgb_submission_v003.csv"))  # C model rows
buildings_v4 = pd.read_csv(os.path.join(sub_dir, "buildings_tweedie_xgb_submission_v004.csv"))  # B model rows

def line_rows(df, prefix):
    out = df[df["row_id"].str[0] == prefix][["row_id", "prediction"]].copy()
    assert out["row_id"].is_unique, f"{prefix} row_id not unique in source file"
    return out

m = line_rows(motor_v3, "M")
c = line_rows(contents_v3, "C")
b = line_rows(buildings_v4, "B")

v10 = pd.concat([m, c, b], ignore_index=True)

# ---- verification ----
assert len(v10) == 626_639, f"unexpected row count: {len(v10):,}"
assert v10["row_id"].is_unique
assert v10["prediction"].isna().sum() == 0 and (v10["prediction"] <= 0).sum() == 0

pref = v10["row_id"].str[0]
print(f"v10 rows: {len(v10):,} | by line: {pref.value_counts().to_dict()}")

for name, d in [("M (motor v003)", m), ("C (contents v003)", c), ("B (buildings v004)", b)]:
    print(f"  {name}: {len(d):,} rows | mean pred {d['prediction'].mean():.2f} | "
          f"min {d['prediction'].min():.2f} | max {d['prediction'].max():.2f}")

v10_path = os.path.join(sub_dir, "motor_buildings_contents_submission_v010.csv")
v10.to_csv(v10_path, index=False)
print(f"\nCombined submission saved to: {v10_path}")
display(v10.head(10))